# Feedforward skip edges and pseudo nodes

This notebook explains how `edge2torch` handles skip edges in the feedforward
backend.

In a feedforward graph, some edges may skip over one or more intermediate
layers. To keep the compiled model strictly layer-wise and efficient,
`edge2torch` expands such skip edges internally into **pseudo nodes**.

These pseudo nodes are:

- internal implementation objects
- used only during compilation and execution
- not exposed in user-facing interpretation outputs

This notebook shows what that means in practice.

## Imports

In [ ]:
import pandas as pd
import torch
from graphviz import Digraph
from IPython.display import display

from edge2torch.compile_graph import compile_graph
from edge2torch.interpret_model import interpret_model

## What problem do pseudo nodes solve?

In a strictly layer-wise feedforward model, each computation block should map
from one layer to the next.

A skip edge such as:

- `gene_1 -> output_1`

breaks that pattern if `output_1` is several layers away from `gene_1`.

To handle this, `edge2torch` expands the skip edge internally into pseudo nodes so
that the compiled graph still connects only adjacent layers.

The figure below illustrates the idea.

![Pseudo nodes explained](../figures/pseudo_nodes_explained.svg)

> **Backend note**
>
> Pseudo nodes are currently implemented only for the `feedforward`
> backend, where they are used to expand skip edges into strictly adjacent
> layer-to-layer computations.
>
> For `recurrent` backends, pseudo nodes are conceivable in principle, but
> they are not currently needed by the implemented compilation strategy and
> are therefore not used.
>
> For `graphnn` backends, pseudo nodes are generally not a natural default,
> because graph neural networks are typically intended to operate directly
> on the original graph topology.

## Step 1: define a graph with a skip edge

We now define a small feedforward graph with one skip edge from the input node
directly to the output node.

This is the kind of graph structure that triggers internal pseudo-node
expansion in the feedforward compiler.

In [ ]:
edgelist = pd.DataFrame(
    {
        "source": [
            "gene_1",
            "pathway_1",
            "pathway_2",
            "gene_1",
        ],
        "target": [
            "pathway_1",
            "pathway_2",
            "output_1",
            "output_1",
        ],
    }
)

edgelist

## Step 2: visualize the original graph

The original graph contains a direct skip edge from `gene_1` to `output_1`.

In [ ]:
dot = Digraph()
dot.attr(rankdir="LR")

input_nodes = ["gene_1"]
layer_1_nodes = ["pathway_1"]
layer_2_nodes = ["pathway_2"]
output_nodes = ["output_1"]

with dot.subgraph() as s:
    s.attr(rank="same")
    for node in input_nodes:
        s.node(node, node)

with dot.subgraph() as s:
    s.attr(rank="same")
    for node in layer_1_nodes:
        s.node(node, node)

with dot.subgraph() as s:
    s.attr(rank="same")
    for node in layer_2_nodes:
        s.node(node, node)

with dot.subgraph() as s:
    s.attr(rank="same")
    for node in output_nodes:
        s.node(node, node)

for _, row in edgelist.iterrows():
    dot.edge(row["source"], row["target"])

dot

## Step 3: compile the graph

We now compile the graph with the feedforward backend.

In [ ]:
model, artifact = compile_graph(
    edgelist=edgelist,
    backend="feedforward",
)

type(model), type(artifact)

## Step 4: inspect the compiled layer structure

The original graph did not contain pseudo nodes explicitly, but the compiled
artifact does. These pseudo nodes are inserted so that the skip edge can be
represented as adjacent layer-to-layer transitions.

In [ ]:
artifact.node_names_by_layer

The pseudo nodes follow names like:

- `pseudo__gene_1__output_1__layer_1`
- `pseudo__gene_1__output_1__layer_2`

These names encode:

- the source node
- the original skip-edge target
- the internal layer where the pseudo node was inserted

## Step 5: inspect the feature names

The original input space is not changed by pseudo-node expansion.

In [ ]:
artifact.feature_names

## Step 6: run the compiled model

Even though pseudo nodes were inserted internally, the compiled model is still
used like an ordinary PyTorch model.

In [ ]:
x = torch.tensor(
    [[0.1], [0.2], [0.3]],
    dtype=torch.float32,
)

y = model(x)
y

In [ ]:
y.shape

## Step 7: interpret the compiled model at node level

Pseudo nodes participate in internal computation, so they are part of the
compiled graph structure.

However, they are implementation-only objects. For that reason, they are
filtered out of the returned node-level interpretation results.

In [ ]:
data = pd.DataFrame(
    {
        "gene_1": [0.1, 0.2, 0.3],
    },
    index=["cell_1", "cell_2", "cell_3"],
)

node_attr = interpret_model(
    model=model,
    artifact=artifact,
    data=data,
    target="nodes",
    method="layer_conductance",
)

node_attr

The returned object is a dictionary of layer-level DataFrames. The biological
nodes are preserved, but the internal pseudo nodes are removed.

In [ ]:
for layer_name, layer_df in node_attr.items():
    print(layer_name)
    display(layer_df)

## Step 8: confirm that pseudo nodes are hidden

Below we compare:

- the internal compiled layer structure
- the returned node-level interpretation outputs

This shows that pseudo nodes exist internally but are not exposed in the
user-facing results.

In [ ]:
compiled_nodes = artifact.node_names_by_layer
returned_nodes = {
    layer_name: list(layer_df.columns)
    for layer_name, layer_df in node_attr.items()
}

compiled_nodes, returned_nodes

## Summary

For the feedforward backend, `edge2torch` expands skip edges internally into pseudo
nodes so that computation remains strictly layer-wise.

This gives three important properties:

1. the original graph can contain skip edges
2. the compiled model remains efficient and layer-wise
3. user-facing interpretation results still expose only the original
   biological nodes, not the internal pseudo nodes